In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
%run 00_utils.ipynb

In [ ]:
def get_std_and_mean_from_dir(root_dir):
    pixel_sum, pixel_squared_sum, total_pixels = 0, 0, 0
    paths = get_all_dir(root_dir)
    for path in paths:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

        img = img.astype(np.float64)
                
        pixel_sum += np.sum(img)
        pixel_squared_sum += np.sum(img ** 2)
        total_pixels += img.size
    mean = pixel_sum / total_pixels
    
    # Varianz berechnen und vor negativen Micro-Werten durch Fließkomma-Ungleichungen schützen
    variance = max(0.0, (pixel_squared_sum / total_pixels) - (mean ** 2))
    std = np.sqrt(variance)
    return mean, std


In [ ]:
# ---------------------------------------------------------
# ANALYSE 2: Pixel-Intensität & Dynamikumfang (IR-Profil)
# ---------------------------------------------------------
def analyze_pixel_intensity(training_dir, num_samples=200):
    print("="*60)
    print(" 4. THERMAL PIXEL INTENSITY & CONTRAST ANALYSIS")
    print("="*60)
    
    img_paths = list(Path(training_dir).rglob("*.jpg")) + list(Path(training_dir).rglob("*.png"))
    if not img_paths:
        print("Keine Bilder für Pixel-Analyse gefunden.")
        return
        
    np.random.seed(42)
    sample_paths = np.random.choice(img_paths, min(num_samples, len(img_paths)), replace=False)
    
    histograms = []
    means, stds = [], []
    
    for p in sample_paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            hist = cv2.calcHist([img], [0], None, [256], [0, 256]).flatten()
            histograms.append(hist)
            means.append(np.mean(img))
            stds.append(np.std(img))
            
    avg_hist = np.mean(histograms, axis=0)
    
    print(f"Stichprobengröße: {len(sample_paths)} Bilder")
    print(f" - Durchschnittliche Pixel-Intensität (Mean): {np.mean(means):.2f}")
    print(f" - Durchschnittliche Standardabweichung (Kontrast): {np.mean(stds):.2f}")
    
    plt.figure(figsize=(10, 4))
    plt.plot(avg_hist, color='black', linewidth=1.5)
    plt.fill_between(range(256), avg_hist, color='gray', alpha=0.5)
    plt.title("Durchschnittliches Graustufen-Histogramm (IR-Intensität)")
    plt.xlabel("Pixelwert (0 = Schwarz, 255 = Weiß)")
    plt.ylabel("Durchschnittliche Pixelanzahl")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

In [ ]:
if __name__ == "__main__":
    mean, std = get_std_and_mean_from_dir(TRAINING_DIR)
    print(f"MEAN: {mean:.2f}   STD: {std:.2f}")
    analyze_pixel_intensity(TRAINING_DIR)